# Workflow: Parallelization

## Load Keys from Env.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from cohere import ClientV2

co = ClientV2(api_key=os.environ["COHERE_API_KEY"])

MODEL_NAME = "command-a-03-2025"

## Define Helper for LLM calls

In [3]:
def llm(prompt: str, system: str = "") -> str:
    msgs = []
    if system:
        msgs.append(
            {
                "role": "system",
                "content": system,
            }
        )
    msgs.append(
        {
            "role": "user",
            "content": prompt,
        }
    )

    return (
        co.chat(
            model=MODEL_NAME,
            messages=msgs,
        )
        .message.content[0]
        .text.strip()
    )

## Describe Specialist Reviewer Prompts

In [5]:
REVIEWERS = {
    "security": "You are a security reviewer. ONLY flag security issues (injection, auth, secrets). Be terse — bullet points.",
    "performance": "You are a performance reviewer. ONLY flag performance issues. Be terse — bullet points.",
    "style": "You are a code-style reviewer. ONLY flag style/readability issues. Be terse — bullet points.",
}

## Input

In [6]:

code_snippet = """
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + user_id
    rows = db.execute(query)
    result = []
    for r in rows:
        result.append(r)
    return result
"""

## Utility to Execute Each Specialist

In [ ]:
def review(name_system):
    name, system = name_system
    return name, llm(f"Review this code:\n\n```python{code_snippet}```", system=system)

## Execute All Specialists In Parallel

In [9]:
from concurrent.futures import ThreadPoolExecutor
import time

t0 = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:
    # Apply the `review` function to each item of the REVIEWERS dictionary.
    reviews = dict(pool.map(review, REVIEWERS.items()))

print(f"All three reviews finished in {time.time() - t0:.1f}s\n")

for name, text in reviews.items():
    print(f"=== {name} ===\n{text}\n")

All three reviews finished in 2.7s

=== security ===
- **SQL Injection**: The code is vulnerable to SQL injection because it directly concatenates `user_id` into the SQL query without proper sanitization or parameterization. An attacker could inject malicious SQL code by manipulating the `user_id` input.  
  **Fix**: Use parameterized queries or prepared statements to safely handle user input.  

Example fix:  
```python
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = %s"
    rows = db.execute(query, (user_id,))
    return list(rows)
```

=== performance ===
- **SQL Injection Vulnerability**: Directly concatenating `user_id` into the query string leaves the code vulnerable to SQL injection attacks. Use parameterized queries instead.  
- **Inefficient Data Handling**: Appending rows to a list in a loop is redundant; directly return the list of rows from the database query.  
- **Unnecessary List Construction**: The `result` list is not needed; return the rows directly

In [11]:
merged = "\n\n".join(f"## {n}\n{t}" for n, t in reviews.items())
summary = llm(
    "Consolidate the following parallel reviews into a single prioritized bullet list "
    "(highest severity first). Don't repeat overlapping findings.\n\n" + merged
)
print("=== Final consolidated review ===\n", summary)

=== Final consolidated review ===
 Here’s a consolidated and prioritized bullet list of findings, with the highest severity issues first, and eliminating overlaps:

**Prioritized Findings:**

- **Critical: SQL Injection Vulnerability**  
  - Directly concatenating `user_id` into the SQL query without sanitization or parameterization exposes the code to SQL injection attacks.  
  **Fix**: Use parameterized queries or prepared statements.  
  **Example**:  
  ```python
  query = "SELECT * FROM users WHERE id = %s"
  rows = db.execute(query, (user_id,))
  ```

- **High: Lack of Error Handling**  
  - No error handling for database query failures or invalid `user_id` inputs, leading to potential crashes or unexpected behavior.  
  **Fix**: Implement try-except blocks to handle exceptions gracefully.

- **Medium: Missing Input Validation**  
  - No checks for `user_id` type or validity, allowing potentially malicious or incorrect inputs.  
  **Fix**: Validate `user_id` before processing (e.

## Advantages

- **Latency**: Wall-clock time is roughly one LLM call, not three.
- **Quality through narrow scopes**: Each reviewer has one job — they don't get distracted defending a security finding while you're asking about style. The aggregator gets a richer, less confused signal to work with.